In [1]:
import pandas as pd

In [2]:
df=pd.read_csv(r'Prod_data\tbl_device_history.csv')

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 299120 entries, 0 to 299119
Data columns (total 14 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   historyId       299120 non-null  int64  
 1   deviceEndpoint  299120 non-null  str    
 2   latitude        299120 non-null  float64
 3   longitude       299120 non-null  float64
 4   trolleyId       299120 non-null  int64  
 5   geozoneId       299120 non-null  int64  
 6   geolayerId      299120 non-null  int64  
 7   createdOn       299120 non-null  str    
 8   modifiedOn      26011 non-null   str    
 9   status          299120 non-null  int64  
 10  nestId          299120 non-null  int64  
 11  wifiRTT         299120 non-null  str    
 12  locationSource  299120 non-null  str    
 13  address         166210 non-null  str    
dtypes: float64(2), int64(6), str(6)
memory usage: 106.4 MB


In [ ]:
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os
load_dotenv()

HOST=os.getenv("DB_HOST")
USER=os.getenv("DB_USER")
PASSWORD=os.getenv("DB_PASSWORD")
DATABASE='demand_forcast_for_trolley'

###=========sqlAlchemy========
engine = create_engine(
    f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}:3306/{DATABASE}"
)

print("Connected using SQLAlchemy!")


        
df = pd.read_sql("SELECT * FROM tbl_device_history", engine)


Connected using SQLAlchemy!
tbl_device_history
   historyId    deviceEndpoint  latitude   longitude  trolleyId  geozoneId  \
0          1  TR1-B43A4535C59C  2.754915  101.703118          1         11   
1          2  TR1-B43A4535C55C  2.754909  101.703125          2         11   
2          3  TR1-B43A4535C59C  2.754895  101.703052          1         11   
3          4  TR1-B43A4535C55C  2.754914  101.703067          2         11   
4          5  TR1-B43A4535C338  2.754914  101.703097          7         11   

   geolayerId            createdOn           modifiedOn  status  nestId  \
0           7  2026-03-04 13:35:04                  NaN       1       0   
1           7  2026-03-04 13:35:48                  NaN       1       0   
2           7  2026-03-04 13:37:14                  NaN       1       0   
3           7  2026-03-04 13:38:05                  NaN       1       0   
4           7  2026-03-04 13:38:57  2026-03-04 14:06:48       1       0   

                                 

In [9]:
df.head(1)

,historyId,deviceEndpoint,latitude,longitude,trolleyId,geozoneId,geolayerId,createdOn,modifiedOn,status,nestId,wifiRTT,locationSource,address
0,1,TR1-B43A4535C59C,2.754915,101.703118,1,11,7,2026-03-04 13:35:04,NaN,1,0,[SSID:BC1-F412FA8990B9 BSSID:F4:12:FA:89:90:B9...,Wi-Fi,"Short Term Car Park 'B', Jalan CTA 2, Sepang, ..."


In [17]:
# Query
query = """
SELECT 
    historyId,
    trolleyId,
    geozoneId,
    geolayerId,
    createdOn,
    address
FROM tbl_device_history
ORDER BY 
    trolleyId ASC,
    STR_TO_DATE(createdOn, '%%d-%%m-%%Y %%H:%%i') ASC;
"""

In [18]:
# Load into DataFrame
df = pd.read_sql(query, engine)


In [19]:
df.head(50)

,historyId,trolleyId,geozoneId,geolayerId,createdOn,address
0,1,1,11,7,2026-03-04 13:35:04,"Short Term Car Park 'B', Jalan CTA 2, Sepang, ..."
1,3,1,11,7,2026-03-04 13:37:14,"Short Term Car Park 'B', Jalan CTA 2, Sepang, ..."
2,8,1,11,7,2026-03-04 13:40:18,"Short Term Car Park 'B', Jalan CTA 2, Sepang, ..."
3,16,1,11,7,2026-03-04 13:42:16,"Short Term Car Park 'B', Jalan CTA 2, Sepang, ..."
4,20,1,11,7,2026-03-04 13:44:23,"Short Term Car Park 'B', Jalan CTA 2, Sepang, ..."
5,23,1,11,7,2026-03-04 13:45:57,"Short Term Car Park 'B', Jalan CTA 2, Sepang, ..."
6,36,1,11,7,2026-03-04 13:51:17,"Short Term Car Park 'B', Jalan CTA 2, Sepang, ..."
7,49,1,0,3,2026-03-04 13:57:33,"Short Term Car Park 'B', Jalan CTA 2, Sepang, ..."
8,56,1,2,3,2026-03-04 14:01:24,"Short Term Car Park 'A', KLIA Departure, Sepan..."
9,62,1,2,3,2026-03-04 14:02:45,"Short Term Car Park 'A', KLIA Departure, Sepan..."


In [ ]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ============================================================
# LOAD DATA
# ============================================================

df = pd.read_sql(
    "SELECT * FROM tbl_device_history",
    engine
)

print("Raw Shape :", df.shape)

# ============================================================
# DATETIME CONVERSION
# ============================================================

df["createdOn"] = pd.to_datetime(
    df["createdOn"],
    format="%d-%m-%Y %H:%M",
    errors="coerce"
)

df = df.dropna(subset=["createdOn"])

# ============================================================
# SORT DATA
# ============================================================

df = (
    df.sort_values(
        ["trolleyId", "createdOn"]
    )
    .reset_index(drop=True)
)

# ============================================================
# DETECT ZONE CHANGES
# ============================================================

df["zone_changed"] = (
    (df["trolleyId"] != df["trolleyId"].shift())
    |
    (df["geozoneId"] != df["geozoneId"].shift())
)

df["stay_id"] = df["zone_changed"].cumsum()

# ============================================================
# BUILD STAYS
# ============================================================

stays = (
    df.groupby("stay_id")
    .agg(
        trolleyId=("trolleyId", "first"),
        geozoneId=("geozoneId", "first"),
        enter_time=("createdOn", "min"),
        exit_time=("createdOn", "max")
    )
    .reset_index(drop=True)
)

stays = stays[
    stays["geozoneId"] != 0
].copy()

print("Stay Records :", len(stays))

# ============================================================
# EXPAND TO HOURLY SLOTS
# ============================================================

min_slot = stays["enter_time"].min().floor("h")
max_slot = stays["exit_time"].max().ceil("h")

all_slots = pd.date_range(
    start=min_slot,
    end=max_slot,
    freq="1h"
)

records = []

for _, row in stays.iterrows():

    for slot_start in all_slots:

        slot_end = slot_start + pd.Timedelta(hours=1)

        if (
            row["enter_time"] < slot_end
            and
            row["exit_time"] > slot_start
        ):

            records.append({
                "trolleyId": row["trolleyId"],
                "geozoneId": row["geozoneId"],
                "slot_start": slot_start,
                "date": slot_start.date(),
                "hour": slot_start.hour,
                "day_num": slot_start.dayofweek,
                "is_weekend": int(slot_start.dayofweek >= 5)
            })

expanded = pd.DataFrame(records)

print("Expanded Shape :", expanded.shape)

# ============================================================
# CREATE TROLLEY COUNT TABLE
# ============================================================

trolley_count = (
    expanded.groupby([
        "date",
        "day_num",
        "is_weekend",
        "hour",
        "slot_start",
        "geozoneId"
    ])["trolleyId"]
    .nunique()
    .reset_index()
    .rename(
        columns={
            "trolleyId": "trolley_count"
        }
    )
)

trolley_count["date"] = pd.to_datetime(
    trolley_count["date"]
)

trolley_count["month"] = (
    trolley_count["date"].dt.month
)

print("Training Dataset Shape :", trolley_count.shape)

# ============================================================
# FEATURE ENGINEERING
# ============================================================

trolley_count["hour_sin"] = np.sin(
    2 * np.pi * trolley_count["hour"] / 24
)

trolley_count["hour_cos"] = np.cos(
    2 * np.pi * trolley_count["hour"] / 24
)

trolley_count["month_sin"] = np.sin(
    2 * np.pi * trolley_count["month"] / 12
)

trolley_count["month_cos"] = np.cos(
    2 * np.pi * trolley_count["month"] / 12
)

trolley_count["day_sin"] = np.sin(
    2 * np.pi * trolley_count["day_num"] / 7
)

trolley_count["day_cos"] = np.cos(
    2 * np.pi * trolley_count["day_num"] / 7
)

# ============================================================
# FEATURES (NO geolayerId)
# ============================================================

FEATURES = [
    "day_num",
    "is_weekend",
    "hour",
    "geozoneId",
    "month",
    "hour_sin",
    "hour_cos",
    "month_sin",
    "month_cos",
    "day_sin",
    "day_cos"
]

TARGET = "trolley_count"

X = trolley_count[FEATURES]
y = trolley_count[TARGET]

# ============================================================
# TRAIN TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

# ============================================================
# RANDOM FOREST TRAINING
# ============================================================

model = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)

print("Training Random Forest...")

model.fit(
    X_train,
    y_train
)

print("Training Completed")

# ============================================================
# EVALUATION
# ============================================================

y_pred = model.predict(X_test)

mae = mean_absolute_error(
    y_test,
    y_pred
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred
    )
)

r2 = r2_score(
    y_test,
    y_pred
)

print("\n==============================")
print("MODEL EVALUATION")
print("==============================")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R2   : {r2:.4f}")

# ============================================================
# FEATURE IMPORTANCE
# ============================================================

importance_df = pd.DataFrame({
    "Feature": FEATURES,
    "Importance": model.feature_importances_
}).sort_values(
    "Importance",
    ascending=False
)

print("\nFeature Importance")
print(importance_df)

# ============================================================
# SAVE MODEL
# ============================================================

joblib.dump(
    model,
    "random_forest_trolley_model_00.pkl"
)

joblib.dump(
    FEATURES,
    "random_forest_features_00.pkl"
)

print("\nModel Saved Successfully")
print("random_forest_trolley_model.pkl")